In [1]:

import scanpy as sc
import numpy as np
import pandas as pd
import warnings

warnings.simplefilter("ignore", category=UserWarning)

import os
os.environ['OPENBLAS_NUM_THREADS'] = '4'   #set threads
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'

sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=True,  #是否添加框线
    figsize=(4, 4)
)


In [2]:
adata_a=sc.read_h5ad('./08_颗粒细胞亚群细分_差异分析/08_after_hb_Gr_subanndata.h5ad')

In [3]:
adata_a

AnnData object with n_obs × n_vars = 33475 × 18144
    obs: 'sample', 'n_genes', 'group', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'S_score', 'G2M_score', 'phase', 'bbknn_leiden_0.5', 'harmony_leiden_0.5', 'harmony_leiden_0.2', 'Major_annotation', 'doublet_score', 'predicted_doublet', 'd_harmony_leiden_0.2', 'Major_annotation_d', 'Gr_subannotation', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'high

###  颗粒细胞各个亚群间差异分析

In [4]:
#adata.uns['log1p']['base'] =None
sc.tl.rank_genes_groups(
    adata_a, "Gr_subannotation", method="wilcoxon",key_added='dea_GC_annotation'
    ,pts=True
    ,use_raw=True  #默认就是用这个
)

/home/a3d9h01010/.local/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [6]:
### 导出差异分析结果
groups=adata_a.uns['dea_GC_annotation']['names'].dtype.names

with pd.ExcelWriter('./08_颗粒细胞亚群细分_差异分析/after_hb_Gr_subannotation_dea_results.xlsx') as writer:
    for group in groups:
        df=sc.get.rank_genes_groups_df(adata_a,group=group,key='dea_GC_annotation')
        df['group']=group
        ### 保存excel表格
        df.to_excel(writer,sheet_name=group)
    



In [7]:
adata_a.obs['Gr_subannotation'].unique()

['GC_4', 'GC_0', 'GC_5', 'GC_1', 'GC_3', 'GC_2']
Categories (6, object): ['GC_0', 'GC_1', 'GC_2', 'GC_3', 'GC_4', 'GC_5']

### 颗粒细胞组间差异分析

In [4]:
cell_group_ids=sorted(adata_a.obs.Gr_subannotation.unique().tolist())

In [6]:
with pd.ExcelWriter('./08_颗粒细胞亚群细分_差异分析/08_each_Gr_group_dea_results.xlsx') as writer:
    for cell_group_id in cell_group_ids:
        print(f'开始{cell_group_id}差异分析!')
        adata_sub=adata_a[adata_a.obs['Gr_subannotation']==cell_group_id].copy()
        key_added_index='dea_'+cell_group_id
        sc.tl.rank_genes_groups(
        adata_sub
        ,groupby='group'
        ,method="wilcoxon"
        ,reference='Control'
        #,groups='PCOS'
        ,key_added=key_added_index
        ,use_raw=True  #默认就是用这个
        )
### 到处差异分析结果
        df=sc.get.rank_genes_groups_df(adata_sub,group=None,key=key_added_index)
        df.to_excel(writer,sheet_name=cell_group_id,index=False)
        print(f'保存完毕{cell_group_id}细胞组间差异分析结果!')

开始GC_0差异分析!
保存完毕GC_0细胞组间差异分析结果!
开始GC_1差异分析!
保存完毕GC_1细胞组间差异分析结果!
开始GC_2差异分析!
保存完毕GC_2细胞组间差异分析结果!
开始GC_3差异分析!
保存完毕GC_3细胞组间差异分析结果!
开始GC_4差异分析!
保存完毕GC_4细胞组间差异分析结果!
开始GC_5差异分析!
保存完毕GC_5细胞组间差异分析结果!
